# 04 群聚調查工作流：從 Line List 到 SitRep

松柏護理之家退伍軍人症群聚事件，長官要求兩小時內交出第一份 SitRep。

這堂課完整走一遍：**讀取 → 摘要指標 → 人時地三要素 → 個案分類 → 結構化輸出**。

In [ ]:
# Google Colab setup -- 若在本機執行可跳過此 cell
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
# --- Step 1: 讀取與資料準備 ---
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("data/synthetic/legionella_outbreak.csv")

# 日期轉換
date_cols = [
    "facility_admission_date", "symptom_onset_date",
    "hospitalization_date", "death_date", "notification_date",
]
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors="coerce")

# 衍生變項
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)
df["age_group"] = pd.cut(
    df["age"], bins=[59, 69, 79, 89, 100],
    labels=["60-69", "70-79", "80-89", "90+"],
)
comorbidity_cols = [
    "comorbidity_chf", "comorbidity_dm",
    "comorbidity_cancer", "comorbidity_copd", "immunosuppressed",
]
df["n_comorbidities"] = df[comorbidity_cols].sum(axis=1)

print(f"資料維度：{df.shape[0]} 筆 × {df.shape[1]} 欄")

In [ ]:
# --- Step 2: 摘要指標 ---
total = len(df)
infected = df["infected"].sum()
confirmed = (df["case_classification"] == "confirmed").sum()
probable = (df["case_classification"] == "probable").sum()
hospitalized = df["hospitalized"].sum()
icu = df["icu_admission"].sum()
deaths = (df["outcome"] == "dead").sum()

print("=" * 50)
print("松柏護理之家退伍軍人症群聚 — SitRep")
print("=" * 50)
print(f"住民總數：{total}")
print(f"感染人數：{infected}（侵襲率 {infected/total:.1%}）")
print(f"  確診：{confirmed}　可能：{probable}")
print(f"住院：{hospitalized}（住院率 {hospitalized/infected:.1%}）")
print(f"ICU：{icu}（ICU 率 {icu/hospitalized:.1%}）")
print(f"死亡：{deaths}（CFR {deaths/infected:.1%}）")

In [ ]:
# --- Step 3: 人 (Person) ---
cases = df[df["infected"] == 1]

print("=== 人口學特徵（感染者）===")
print(f"年齡中位數：{cases['age'].median():.0f} 歲"
      f"（範圍 {cases['age'].min()}-{cases['age'].max()}）")
print(f"男性比例：{(cases['sex'] == 'M').mean():.1%}")

print(f"\n--- 年齡組分布 ---")
age_dist = cases["age_group"].value_counts().sort_index()
for grp, n in age_dist.items():
    print(f"  {grp}: {n} ({n/len(cases):.1%})")

print(f"\n--- 共病分布 ---")
for col in comorbidity_cols:
    label = col.replace("comorbidity_", "").upper()
    n = int(cases[col].sum())
    print(f"  {label}: {n} ({n/len(cases):.1%})")

In [ ]:
# --- Step 4: 時 (Time) — 流行曲線 ---
daily = cases.groupby("symptom_onset_date").size().rename("cases")

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(daily.index, daily.values, color="#2c7fb8", edgecolor="white")
ax.set_title("流行曲線（依發病日）", fontsize=14)
ax.set_xlabel("發病日期")
ax.set_ylabel("新增病例數")
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

print(f"流行期間：{daily.index.min().date()} – {daily.index.max().date()}")
print(f"高峰日：{daily.idxmax().date()}（{daily.max()} 例）")

In [ ]:
# --- Step 5: 地 (Place) — 各翼區侵襲率 ---
wing_stats = (
    df.groupby(["floor", "wing"])
    .agg(
        residents=("case_id", "size"),
        infected=("infected", "sum"),
        deaths=("outcome", lambda x: (x == "dead").sum()),
    )
    .reset_index()
)
wing_stats["AR%"] = (wing_stats["infected"] / wing_stats["residents"] * 100).round(1)
wing_stats["CFR%"] = (wing_stats["deaths"] / wing_stats["infected"] * 100).round(1)
wing_stats["label"] = wing_stats["floor"].astype(str) + wing_stats["wing"]

print("=== 各翼區疫情摘要 ===")
print(wing_stats[["label", "residents", "infected", "AR%", "deaths", "CFR%"]]
      .to_string(index=False))

In [ ]:
# --- Step 6: 個案分類分層摘要 ---
classification = (
    df.groupby("case_classification")
    .agg(
        n=("case_id", "size"),
        hospitalized=("hospitalized", "sum"),
        icu=("icu_admission", "sum"),
        deaths=("outcome", lambda x: (x == "dead").sum()),
    )
)
classification["hosp_rate%"] = (
    classification["hospitalized"] / classification["n"] * 100
).round(1)

print("=== 按個案分類分層 ===")
print(classification.to_string())

In [ ]:
# --- Step 7: 包成可重跑函式 ---
def generate_sitrep(csv_path):
    """從 CSV 產出 SitRep 摘要字典。"""
    df = pd.read_csv(csv_path)
    for col in ["symptom_onset_date", "hospitalization_date",
                "death_date", "notification_date"]:
        df[col] = pd.to_datetime(df[col], errors="coerce")
    df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)

    total = len(df)
    infected = int(df["infected"].sum())
    deaths = int((df["outcome"] == "dead").sum())

    return {
        "total_residents": total,
        "infected": infected,
        "attack_rate": round(infected / total * 100, 1),
        "deaths": deaths,
        "cfr": round(deaths / infected * 100, 1) if infected else 0,
        "hospitalized": int(df["hospitalized"].sum()),
        "icu": int(df["icu_admission"].sum()),
    }

sitrep = generate_sitrep("data/synthetic/legionella_outbreak.csv")
print("=== 結構化 SitRep 輸出 ===")
for k, v in sitrep.items():
    print(f"  {k}: {v}")

## 小結

你已經完成一份標準 SitRep 的自動化產出流程：

| 步驟 | 內容 | 學到的技能 |
|------|------|------------|
| 1 | 讀取與準備 | 日期轉換、衍生變項 |
| 2 | 摘要指標 | 侵襲率、CFR、住院率 |
| 3 | 人 | 年齡分布、共病盛行率 |
| 4 | 時 | 流行曲線、高峰日 |
| 5 | 地 | 翼區侵襲率表 |
| 6 | 個案分類 | 確診/可能/非個案分層 |
| 7 | 函式化 | 可重跑的 `generate_sitrep()` |

下一章（Ch05），我們回頭處理 Ch03 留下的問題——淋浴使用的高 RR 是真的因果關係，還是被交絡因子膨脹了？